# E10: PLC-CECT dataset audit and counterfactual shortcut test

**Revision plan, 00 log section 24 (approved 18 Oct 2026).** The paper will only claim what this notebook proves.

**Version 2 (19 Oct 2026):** the first run stopped on a real data finding — for one patient the tumour mask and the liver mask have different numbers of slices (132 vs 133). Version 2 handles masks on another grid exactly as Week 2 did (no in-plane window for that scan), the census reports every such file, and a problem in one patient is recorded and listed instead of stopping everything. Nothing else changed.
Nothing is trained or selected here; the main models are not changed by the results.

- **E10a — census (facts):** every CT, tumour-mask and liver-mask file of the 278 internal patients (4 phases): shape, slices,
  storage format (3-D or 3-channel), data type, header voxel size, intensity range; for 3-channel files, whether the channels
  are identical, and where they differ (inside the liver / tumour or not).
- **E10b — slice thickness or scan range (anatomical estimate):** slices containing liver ÷ liver width in pixels, by class.
  Rule fixed in the plan: class medians differing > 1.5× with similar liver coverage → thickness; similar ratio but different
  slice counts → scan range; otherwise mixed ("similar" is operationalised as max/min ≤ 1.25 for this notebook).
- **E10c — counterfactual test with the frozen models:** each scan is resampled along the slice axis only, the crops are rebuilt
  exactly as Week 2 did, and the frozen phase-dropout CNN, E2 radiomics model and CNN + radiomics fusion predict again
  (development patients with their own fold model, held-out patients with the 5-model ensemble). Conditions: unchanged
  (must reproduce the saved results), **slice-density swap** HCC ↔ ICC (× 195/93 and × 93/195, all phases), **non-contrast
  artefact injected** into HCC/ICC (non-contrast slices = 0.29 × portal-venous slices) and **repaired** in cHCC-CCA
  (non-contrast slices = portal-venous slices). Each is compared with its own **smoothing control** (the same resampling and
  back to the original number of slices), which separates geometry from interpolation blur.
- **Rules (fixed in the plan):** reliance is claimed only if the shift goes in the predicted direction, its bootstrap 95% CI
  excludes 0 and the mean shift is ≥ 0.05 — for slice density in both directions (HCC→ICC-like and ICC→HCC-like), for the
  artefact both when injected and when repaired. Smoothing controls must move the tested score by < 0.05 on average.
- **E10d — evidence images** for the supplement (patients chosen by a fixed seed, not by eye).

**Runtime:** GPU runtime with High-RAM (L4 or A100), about 1.5–2 hours (reading all 1,112 internal scans from Drive,
resampling, radiomics on the CPU cores). No training. **Safety:** inputs are only read; outputs go to a new folder
`liver_project/revision_2026-10/E10_<time>/`. The last cell saves everything to Drive and shuts the runtime down
(set `SHUTDOWN_WHEN_DONE = False` there to keep it).

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder
import os, sys, json, time, hashlib, platform, datetime, warnings, subprocess, tarfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    WORK = Path('/content/e10_work')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm==4.6.0', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E10_TEST_ROOT'])
    WORK = Path(os.environ['E10_TEST_WORK'])
WORK.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import joblib
import nibabel as nib
import SimpleITK as sitk
import lightgbm as lgb
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy import ndimage
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
E8B_DIR = REVISION_DIR / 'E8b_20261018T025705Z'
SEED, N_BOOT = 42, 2000
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
CLASSES = ['HCC', 'ICC', 'CHCC']
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
CROP = 96
# fixed in the plan (00 log section 24): portal-venous median slices HCC 93, ICC 195 (E9a); cHCC-CCA non-contrast ratio 0.29
SWAP_FACTOR = {'HCC': 195 / 93, 'ICC': 93 / 195}
INJECT_RATIO = 0.29
MIN_SHIFT = 0.05
INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e2_radiomics_oof': (E2_DIR / 'E2_radiomics_noshape_oof_predictions.csv', '21057b42933293a7d45cbefbb6f58dfe89458064a42fff689abcbc826fe28cb6'),
    'e2_radiomics_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e8b_predictions': (E8B_DIR / 'E8b_E9_predictions.csv', '644a520c0d034ce233bb88ab98e1b6d9e89993a661e380d06985edde4aa34064'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {**{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
               **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)}}
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}'); continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems.append(f'missing or changed size: {tar_path}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CPU = os.cpu_count() or 2
N_JOBS = max(1, min(8, N_CPU - 1)) if IN_COLAB else 1
N_IO = 3 if IN_COLAB else 2
import sklearn, monai, radiomics
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'lightgbm': lgb.__version__,
            'pyradiomics': radiomics.__version__, 'nibabel': nib.__version__, 'SimpleITK': sitk.Version_VersionString(),
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu',
            'cpus': N_CPU}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E10_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES), '| versions:', VERSIONS, '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Data: split, folds, labels, the Week 2 processed crops (for the reproduction gate), patient table
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E10_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
ALL_IDS = dev_ids + test_ids
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
FOLD_OF = {str(p): f for f, s in enumerate(fold_splits) for p in s['val_ids']}
assert set(FOLD_OF) == set(dev_ids), 'fold files do not cover the development set exactly'
internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
LABEL = internal.set_index('patient_id')['label'].astype(int).to_dict()
CLS = {p: CLASSES[LABEL[p]] for p in ALL_IDS}
table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
table = table[table['patient_id'].isin(ALL_IDS)].drop_duplicates(['patient_id', 'phase'])          # Week 2: first row per phase
ROWS = {pid: g.set_index('phase') for pid, g in table.groupby('patient_id')}
missing = [p for p in ALL_IDS if p not in ROWS or any(ph not in ROWS[p].index for ph in PHASE_ORDER)]
if missing:
    raise RuntimeError(f'STOP — patients without all four phases: {missing[:10]}')
nums = pd.Series({p: int(''.join(ch for ch in p if ch.isdigit())) for p in ALL_IDS})
BLOCKS = {c: (int(nums[[p for p in ALL_IDS if CLS[p] == c]].min()), int(nums[[p for p in ALL_IDS if CLS[p] == c]].max())) for c in CLASSES}
print('Patients:', len(ALL_IDS), '| classes', pd.Series(CLS).value_counts().to_dict(), '| patient-number blocks by class:', BLOCKS)

In [ ]:
# Cell 3 — E10a census + E10b anatomy + E10c crops for every condition, in one pass over the original scans
def load_nii(path):                          # Week 2, unchanged (returns the array exactly as Week 2 used it)
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

def week2_crop(ct, tumor, liver):             # Week 2 process_patient, one phase, unchanged
    ct_masked = ct * (liver > 0).astype(np.float32)
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    return np.clip(crop_cube(ct_masked, center, CROP), 0, 255).astype(np.uint8), (crop_cube(tumor, center, CROP) > 0)

def resample_slices(vol, n_new, is_mask):
    """Change only the number of slices (axis 2). CT linear; masks linear then > 0.5."""
    n_new = max(2, int(round(n_new)))
    if n_new == vol.shape[2]:
        return vol
    out = ndimage.zoom(vol.astype(np.float32), (1.0, 1.0, n_new / vol.shape[2]), order=1)
    assert out.shape[2] == n_new, (out.shape, n_new)
    return (out > 0.5).astype(np.float32) if is_mask else out

def resampled(phase_vols, n_new, back=False):
    ct, tm, lv = phase_vols
    n0 = ct.shape[2]
    out = [resample_slices(ct, n_new, False), resample_slices(tm, n_new, True), resample_slices(lv, n_new, True)]
    if back:
        out = [resample_slices(out[0], n0, False), resample_slices(out[1], n0, True), resample_slices(out[2], n0, True)]
    return out

def file_facts(img, arr_raw, liver, tumor):
    shape = tuple(int(s) for s in img.shape)
    facts = {'shape': str(shape), 'ndim': len(shape), 'channels': int(shape[-1]) if len(shape) == 5 else 1,
             'dtype': str(img.get_data_dtype()), 'header_voxel_size': str(tuple(round(float(z), 4) for z in img.header.get_zooms()[:3])),
             'min': float(arr_raw.min()), 'max': float(arr_raw.max()), 'integer_valued': bool(np.all(np.mod(arr_raw, 1) == 0))}
    if len(shape) == 5 and shape[-1] == 3:
        ch = arr_raw[..., 0, :]
        diff = (ch[..., 0] != ch[..., 1]) | (ch[..., 1] != ch[..., 2])
        facts.update({'channels_identical': bool(not diff.any()), 'voxels_channels_differ': int(diff.sum()),
                      'differ_inside_liver': int((diff & (liver > 0)).sum()) if liver.shape == diff.shape else -1,
                      'differ_inside_tumour': int((diff & (tumor > 0)).sum()) if tumor.shape == diff.shape else -1})
    return facts

def extent(mask, axis):
    idx = np.where(mask.any(axis=tuple(a for a in range(3) if a != axis)))[0]
    return (int(idx.min()), int(idx.max())) if idx.size else (-1, -1)

def process_patient(pid):
    rows = ROWS[pid]
    vols, census, anat = {}, [], []
    for ph in PHASE_ORDER:
        img = nib.load(str(DATA_DIR / rows.loc[ph, 'ct_path']))
        raw = img.get_fdata().astype(np.float32)                    # the same call Week 2 made, kept unreduced for the census
        ct = raw
        while ct.ndim > 3:                                          # Week 2 load_nii reduction, unchanged
            ct = ct.squeeze(-1) if ct.shape[-1] == 1 else (ct.mean(axis=-1) if ct.shape[-1] == 3 else ct[..., 0])
        tm_img, lv_img = nib.load(str(DATA_DIR / rows.loc[ph, 'mask_path'])), nib.load(str(DATA_DIR / rows.loc[ph, 'liver_mask_path']))
        tumor, liver = load_nii(DATA_DIR / rows.loc[ph, 'mask_path']), load_nii(DATA_DIR / rows.loc[ph, 'liver_mask_path'])
        f = file_facts(img, raw, liver, tumor)
        census.append({'patient_id': pid, 'class': CLS[pid], 'phase': ph, **{f'ct_{k}': v for k, v in f.items()},
                       'tumour_mask_shape': str(tuple(int(s) for s in tm_img.shape)), 'liver_mask_shape': str(tuple(int(s) for s in lv_img.shape)),
                       'tumour_mask_values': str(sorted(np.unique(tumor).astype(float).tolist())[:6]),
                       'liver_mask_values': str(sorted(np.unique(liver).astype(float).tolist())[:6]),
                       'masks_match_ct_grid': bool(tumor.shape == ct.shape and liver.shape == ct.shape)})
        lz, li, lj = extent(liver > 0, 2), extent(liver > 0, 0), extent(liver > 0, 1)
        tz, ti, tj = extent(tumor > 0, 2), extent(tumor > 0, 0), extent(tumor > 0, 1)
        n = ct.shape[2]
        anat.append({'patient_id': pid, 'class': CLS[pid], 'phase': ph, 'n_slices': n, 'liver_slices': int((liver > 0).any(axis=(0, 1)).sum()),
                     'liver_width_px': max(li[1] - li[0] + 1, lj[1] - lj[0] + 1), 'liver_touches_first_or_last_slice': bool(lz[0] == 0 or lz[1] == n - 1),
                     'tumour_slices': tz[1] - tz[0] + 1 if tz[0] >= 0 else 0,
                     'tumour_inplane_px': max(ti[1] - ti[0] + 1, tj[1] - tj[0] + 1) if ti[0] >= 0 else 0})
        # in-plane window around liver and tumour: outside it, liver-masked CT and the tumour mask are zero, so the crops built from
        # the window are identical to Week 2's (checked by gate A); only the slice axis is ever resampled, so it stays complete
        win = None
        if tumor.shape == ct.shape and liver.shape == ct.shape:     # masks on another grid (found in the first run): no window
            both = (tumor > 0) | (liver > 0)
            ii, jj = np.where(both.any(axis=(1, 2)))[0], np.where(both.any(axis=(0, 2)))[0]
            if ii.size and jj.size:
                win = (slice(int(ii.min()), int(ii.max()) + 1), slice(int(jj.min()), int(jj.max()) + 1))
            del both
        vols[ph] = tuple(np.ascontiguousarray(a[win]) for a in (ct, tumor, liver)) if win else (ct, tumor, liver)   # else: full arrays, as Week 2
        del raw, ct, tumor, liver
    c = CLS[pid]
    plan = {'control': {}}                                          # condition -> {phase: (new slice count, back to original?)}
    if c in ('HCC', 'ICC'):
        plan['swap'] = {ph: (vols[ph][0].shape[2] * SWAP_FACTOR[c], False) for ph in PHASE_ORDER}
        plan['swap_smooth'] = {ph: (vols[ph][0].shape[2] * SWAP_FACTOR[c], True) for ph in PHASE_ORDER}
        plan['inject'] = {'P': (INJECT_RATIO * vols['C2'][0].shape[2], False)}
        plan['inject_smooth'] = {'P': (INJECT_RATIO * vols['C2'][0].shape[2], True)}
    else:
        plan['repair'] = {'P': (vols['C2'][0].shape[2], False)}
        plan['repair_smooth'] = {'P': (vols['C2'][0].shape[2], True)}
    arrays = {}
    for name, changes in plan.items():                              # one condition at a time (memory)
        cts, tms, ns = [], [], []
        for ph in PHASE_ORDER:
            v = resampled(vols[ph], changes[ph][0], back=changes[ph][1]) if ph in changes else vols[ph]
            ct4, tm4 = week2_crop(*v)
            cts.append(ct4); tms.append(tm4); ns.append(v[0].shape[2])
            del v
        arrays[f'{name}_ct'], arrays[f'{name}_tm'], arrays[f'{name}_nslices'] = np.stack(cts), np.stack(tms), np.array(ns)
    np.savez_compressed(WORK / f'{pid}.npz', **arrays)
    return pid, census, anat, list(plan)

def ram_text():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f' | RAM {vm.used / 1e9:.0f}/{vm.total / 1e9:.0f} GB'
    except Exception:
        return ''

def safe_process(pid):                                              # a data problem in one patient is recorded, not fatal
    try:
        return process_patient(pid) + (None,)
    except Exception as error:
        return pid, [], [], [], f'{type(error).__name__}: {error}'

t0 = time.time()
CENSUS, ANAT, CONDS, FAILED = [], [], {}, {}
N_PROC = max(1, min(6, N_CPU - 2)) if IN_COLAB else 1
def collect(results):
    for n, (pid, census, anat, conds, error) in enumerate(results, start=1):
        if error:
            FAILED[pid] = error; print(f'  {pid}: {error}', flush=True)
        else:
            CENSUS.extend(census); ANAT.extend(anat); CONDS[pid] = conds
        if n % 25 == 0 or n == len(ALL_IDS):
            print(f'  patients processed {n}/{len(ALL_IDS)} ({(time.time() - t0) / 60:.1f} min){ram_text()}', flush=True)
done = False
if N_PROC > 1:
    try:                                                            # separate processes: resampling does not wait for one another
        collect(joblib.Parallel(n_jobs=N_PROC, return_as='generator')(joblib.delayed(safe_process)(p) for p in ALL_IDS))
        done = True
    except Exception as error:                                     # only infrastructure problems end up here
        print(f'  parallel processing failed ({error!r:.120}); continuing with threads')
        CENSUS.clear(); ANAT.clear(); CONDS.clear(); FAILED.clear()
if not done:
    with ThreadPoolExecutor(N_IO) as pool:
        collect(pool.map(safe_process, ALL_IDS))
if FAILED:
    pd.Series(FAILED).to_csv(OUT_DIR / 'E10_failed_patients.csv')
    raise RuntimeError(f'STOP — {len(FAILED)} patients could not be processed (listed in E10_failed_patients.csv): {dict(list(FAILED.items())[:5])}')
CENSUS, ANAT = pd.DataFrame(CENSUS), pd.DataFrame(ANAT)
CENSUS.to_csv(OUT_DIR / 'E10a_census_per_file.csv', index=False)
ANAT.to_csv(OUT_DIR / 'E10b_anatomy_per_scan.csv', index=False)

# ---- gate A: the unchanged condition must reproduce the Week 2 processed crops ----
same = 0
for pid in ALL_IDS:
    with np.load(PROCESSED_DIR / f'{pid}.npz') as z, np.load(WORK / f'{pid}.npz') as w:
        same += int(np.array_equal(z['ct'].astype(np.uint8), w['control_ct']))
GATE = {'control_crops_identical_to_processed': f'{same}/{len(ALL_IDS)}'}
print('Gate A — unchanged crops identical to the Week 2 processed crops:', GATE['control_crops_identical_to_processed'])
if IN_COLAB and same < 0.99 * len(ALL_IDS):
    raise RuntimeError('STOP — the rebuilt crops do not reproduce the Week 2 crops. Send this output.')

In [ ]:
# Cell 4 — E10a census summary and E10b slice thickness or scan range (rule fixed in the plan)
cs = CENSUS.copy()
cs['three_channel'] = cs['ct_channels'] == 3
SUMMARY_A = cs.groupby('class').agg(files=('patient_id', 'size'), three_channel_files=('three_channel', 'sum'),
                                    patients_with_three_channel=('patient_id', lambda s: int(cs.loc[s.index[cs.loc[s.index, 'three_channel'].to_numpy()], 'patient_id'].nunique())),
                                    dtypes=('ct_dtype', lambda s: ', '.join(sorted(set(s)))), voxel_sizes=('ct_header_voxel_size', lambda s: ', '.join(sorted(set(s)))),
                                    min_value=('ct_min', 'min'), max_value=('ct_max', 'max'), all_integer=('ct_integer_valued', 'all'),
                                    masks_on_ct_grid=('masks_match_ct_grid', 'mean'))
tc = cs[cs['three_channel']]
if len(tc):
    SUMMARY_A = SUMMARY_A.join(tc.groupby('class').agg(three_channel_identical=('ct_channels_identical', 'mean'),
                                                        differ_inside_liver_files=('ct_differ_inside_liver', lambda s: int((s > 0).sum())),
                                                        differ_inside_tumour_files=('ct_differ_inside_tumour', lambda s: int((s > 0).sum()))))
SUMMARY_A.to_csv(OUT_DIR / 'E10a_census_by_class.csv')
print('E10a — census by class:\n', SUMMARY_A.to_string())

an = ANAT.copy()
an['liver_mask_empty'] = an['liver_slices'] == 0                       # found in the first v2 run: 13 scans of 4 ICC patients
an['tumour_mask_empty'] = an['tumour_slices'] == 0
EMPTY_MASKS = an.loc[an['liver_mask_empty'] | an['tumour_mask_empty'], ['patient_id', 'class', 'phase', 'liver_mask_empty', 'tumour_mask_empty']]
EMPTY_MASKS = EMPTY_MASKS.assign(split=EMPTY_MASKS['patient_id'].map(lambda p: 'held-out' if p in set(test_ids) else 'development'))
EMPTY_MASKS.to_csv(OUT_DIR / 'E10a_empty_masks.csv', index=False)
print('E10a — scans with an empty liver or tumour mask (Week 2 crops of an empty liver mask are all zero):')
print(EMPTY_MASKS.to_string(index=False))
valid = ~an['liver_mask_empty']                                          # empty liver masks are excluded from the anatomy estimate
an['ratio'] = np.where(valid, an['liver_slices'] / an['liver_width_px'].clip(lower=1), np.nan)
an['coverage'] = np.where(valid, an['liver_slices'] / an['n_slices'], np.nan)
an['liver_touches_first_or_last_slice'] = np.where(valid, an['liver_touches_first_or_last_slice'].astype(float), np.nan)
SUMMARY_B = an.groupby(['phase', 'class']).agg(median_slices=('n_slices', 'median'), median_liver_slices=('liver_slices', 'median'),
                                               median_ratio=('ratio', 'median'), median_coverage=('coverage', 'median'),
                                               liver_truncated=('liver_touches_first_or_last_slice', 'mean'))
SIMILAR, DIFFERENT = 1.25, 1.5          # "similar" operationalised as max/min <= 1.25; rule threshold 1.5 from the plan
verdicts = {}
for ph in PHASE_ORDER:
    s = SUMMARY_B.loc[ph]
    spread = lambda col: float(s[col].max() / max(s[col].min(), 1e-9))
    r, cov, sl = spread('median_ratio'), spread('median_coverage'), spread('median_slices')
    verdict = ('slice thickness' if r > DIFFERENT and cov <= SIMILAR else
               'scan range' if r <= SIMILAR and sl > SIMILAR else
               'no class difference' if r <= SIMILAR and sl <= SIMILAR else 'mixed')
    verdicts[ph] = {'ratio_spread': round(r, 2), 'coverage_spread': round(cov, 2), 'slices_spread': round(sl, 2), 'verdict': verdict}
# cHCC-CCA non-contrast vs its own portal-venous scan (within patient)
w = an.pivot(index='patient_id', columns='phase', values=['n_slices', 'liver_slices', 'liver_touches_first_or_last_slice']).astype(float)
within = pd.DataFrame({'class': pd.Series(CLS), 'slices_P_over_C2': w[('n_slices', 'P')] / w[('n_slices', 'C2')],
                       'liver_slices_P_over_C2': w[('liver_slices', 'P')].replace(0, np.nan) / w[('liver_slices', 'C2')].replace(0, np.nan),
                       'liver_truncated_in_P': w[('liver_touches_first_or_last_slice', 'P')]})
WITHIN = within.groupby('class').agg(median_slices_P_over_C2=('slices_P_over_C2', 'median'),
                                     median_liver_slices_P_over_C2=('liver_slices_P_over_C2', 'median'),
                                     liver_truncated_in_P=('liver_truncated_in_P', 'mean'))
SUMMARY_B.to_csv(OUT_DIR / 'E10b_anatomy_by_class.csv'); WITHIN.to_csv(OUT_DIR / 'E10b_noncontrast_within_patient.csv')
print('E10b — anatomy by phase and class:\n', SUMMARY_B.round(3).to_string())
print('E10b — verdict per phase (rule fixed in the plan):', json.dumps(verdicts, indent=1))
print('E10b — non-contrast vs portal-venous within patient:\n', WITHIN.round(3).to_string())

In [ ]:
# Cell 5 — Frozen phase-dropout CNN on every condition (own fold model for development patients, ensemble for held-out); gate B
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

D_MODELS = [load_state(MODEL_FILES[f'd_{f}']) for f in range(5)]

@torch.no_grad()
def cnn_predict(pid, ct4):
    x = torch.from_numpy(zscore4(ct4.astype(np.float32))).unsqueeze(0).to(DEVICE)
    models = [D_MODELS[FOLD_OF[pid]]] if pid in FOLD_OF else D_MODELS
    return np.mean([torch.softmax(m(x).float(), dim=1).cpu().numpy()[0] for m in models], axis=0)

rows = []
for pid in ALL_IDS:
    with np.load(WORK / f'{pid}.npz') as w:
        for cond in CONDS[pid]:
            rows.append({'model': 'CNN', 'patient_id': pid, 'condition': cond, **dict(zip(P3, cnn_predict(pid, w[f'{cond}_ct'])))})
PRED = pd.DataFrame(rows)
saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
saved = saved[saved['pattern'] == 'complete']
ref = pd.concat([saved[saved['split'] == 'oof'].set_index('patient_id').loc[dev_ids, P3],
                 saved[saved['split'] == 'test'].set_index('patient_id').loc[test_ids, P3]])
ours = PRED[(PRED['model'] == 'CNN') & (PRED['condition'] == 'control')].set_index('patient_id').loc[ALL_IDS, P3]
GATE['cnn_control_vs_E3'] = float(np.abs(ours.to_numpy() - ref.loc[ALL_IDS].to_numpy()).max())
print('Gate B — unchanged CNN predictions vs saved E3 (max difference, must be < 0.02):', GATE['cnn_control_vs_E3'])
if IN_COLAB and GATE['cnn_control_vs_E3'] > 0.02:
    raise RuntimeError('STOP — the frozen CNN does not reproduce E3. Send this output.')

In [ ]:
# Cell 6 — Frozen E2 radiomics model on every condition (Week 5 extraction settings, unchanged); gate C (reported)
RADIOMICS_PARAMS = {                          # Week 5, unchanged
    'setting': {'binWidth': 5, 'label': 1, 'force2D': False, 'normalize': False, 'resampledPixelSpacing': None},
    'imageType': {'Original': {}},
    'featureClass': {'firstorder': None, 'shape': None, 'glcm': None, 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None},
}
def radiomics_patient(pid, conds, work_dir):
    extractor = featureextractor.RadiomicsFeatureExtractor(RADIOMICS_PARAMS)
    out = {}
    with np.load(work_dir / f'{pid}.npz') as w:
        for cond in conds:
            ct4, tm4 = w[f'{cond}_ct'], w[f'{cond}_tm']
            feats = {}
            for i, phase in enumerate(PHASE_ORDER):
                try:
                    result = extractor.execute(sitk.GetImageFromArray(ct4[i].astype(np.float32)), sitk.GetImageFromArray(tm4[i].astype(np.uint8)))
                except Exception:
                    continue
                for key, value in result.items():
                    if not key.startswith('diagnostics_'):
                        try:
                            feats[f'{phase}_{key}'] = float(value)
                        except (TypeError, ValueError):
                            pass
            out[cond] = feats
    return pid, out

t0 = time.time()
jobs = [(pid, CONDS[pid], WORK) for pid in ALL_IDS]
res = None
if N_JOBS > 1:
    try:
        res = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(radiomics_patient)(*job) for job in jobs)
    except Exception as error:
        print(f'  parallel radiomics failed ({error!r:.120}); continuing one patient at a time')
if res is None:
    res = [radiomics_patient(*job) for job in jobs]
RADF = pd.DataFrame([{'patient_id': pid, 'condition': cond, **feats} for pid, d in res for cond, feats in d.items()])
RADF.to_csv(OUT_DIR / 'E10c_radiomics_features.csv.gz', index=False, compression='gzip')
print(f'Radiomics extracted: {RADF.shape} ({(time.time() - t0) / 60:.1f} min)')

rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]
POOL = rad_models[0]['feature_pool']
def rad_predict(pid, row):
    X = row.reindex(POOL).to_numpy(dtype=np.float64)[None, :]
    models = [rad_models[FOLD_OF[pid]]] if pid in FOLD_OF else rad_models
    return np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X[:, m['sel_idx']].astype(np.float32))))[0] for m in models], axis=0)
rows = []
for _, r in RADF.iterrows():
    rows.append({'model': 'radiomics', 'patient_id': r['patient_id'], 'condition': r['condition'], **dict(zip(P3, rad_predict(r['patient_id'], r)))})
PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True)

ctrl = RADF[RADF['condition'] == 'control'].set_index('patient_id').loc[ALL_IDS]
ref_feat = internal.set_index('patient_id').loc[ALL_IDS, POOL]
rel = (ctrl.reindex(columns=POOL).astype(float) - ref_feat.astype(float)).abs() / (ref_feat.astype(float).abs() + 1e-6)
ref_rad = pd.concat([pd.read_csv(INPUTS['e2_radiomics_oof'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids, P3],
                     pd.read_csv(INPUTS['e2_radiomics_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, P3]])
ours = PRED[(PRED['model'] == 'radiomics') & (PRED['condition'] == 'control')].set_index('patient_id').loc[ALL_IDS, P3]
GATE['radiomics_features_vs_week5_max_relative_difference'] = float(np.nanmax(rel.to_numpy()))
GATE['radiomics_control_vs_E2'] = float(np.abs(ours.to_numpy() - ref_rad.loc[ALL_IDS].to_numpy()).max())
print('Gate C — unchanged radiomics: features vs Week 5 (max relative difference):', GATE['radiomics_features_vs_week5_max_relative_difference'],
      '| predictions vs saved E2 (max difference):', GATE['radiomics_control_vs_E2'])
if GATE['radiomics_control_vs_E2'] > 0.02:
    print('NOTE — the re-extracted features do not reproduce Week 5 exactly; every radiomics comparison below is between conditions '
          'extracted here in the same way, so the shifts remain valid, but this is reported.')

In [ ]:
# Cell 7 — Frozen CNN + radiomics meta-learner (the reference fusion of E8b, refitted on the saved out-of-fold inputs); gate D
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
oof_meta = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids].reset_index()
d_saved = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_oof = d_saved.query("pattern == 'complete' and split == 'oof'").set_index('patient_id').loc[dev_ids, P3]
oof_meta[W4] = d_oof.loc[oof_meta['patient_id']].to_numpy()

def prepare(X):                               # E8b, unchanged
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X
FITTED = []
for f in range(5):                            # E8b meta_folds, unchanged
    tr = oof_meta['fold'].astype(int).ne(f).to_numpy()
    X = prepare(oof_meta.loc[tr, W4 + W5].to_numpy())
    imp = SimpleImputer(strategy='median').fit(X)
    sc = StandardScaler().fit(imp.transform(X))
    m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof_meta.loc[tr, 'label'].astype(int))
    FITTED.append((imp, sc, m))
cnn = PRED[PRED['model'] == 'CNN'].set_index(['patient_id', 'condition'])[P3]
rad = PRED[PRED['model'] == 'radiomics'].set_index(['patient_id', 'condition'])[P3]
rows = []
for (pid, cond) in cnn.index:
    if (pid, cond) not in rad.index:
        continue
    x = prepare(np.concatenate([cnn.loc[(pid, cond)].to_numpy(), rad.loc[(pid, cond)].to_numpy()])[None, :])
    fitted = [FITTED[FOLD_OF[pid]]] if pid in FOLD_OF else FITTED
    p = np.mean([mm.predict_proba(s.transform(i.transform(x)))[0] for i, s, mm in fitted], axis=0)
    rows.append({'model': 'CNN + radiomics', 'patient_id': pid, 'condition': cond, **dict(zip(P3, p))})
PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True)
PRED.to_csv(OUT_DIR / 'E10c_predictions.csv', index=False)
e8b = pd.read_csv(INPUTS['e8b_predictions'][0], dtype={'patient_id': str})
e8b = e8b[e8b['model'] == 'cnn_plus_radiomics: [ref] phase_dropout_CNN + radiomics_noshape']
ref_fus = pd.concat([e8b[e8b['cohort'] == 'internal_oof'].set_index('patient_id').loc[dev_ids, P3],
                     e8b[e8b['cohort'] == 'internal_test'].set_index('patient_id').loc[test_ids, P3]])
ours = PRED[(PRED['model'] == 'CNN + radiomics') & (PRED['condition'] == 'control')].set_index('patient_id').loc[ALL_IDS, P3]
GATE['fusion_control_vs_E8b'] = float(np.abs(ours.to_numpy() - ref_fus.loc[ALL_IDS].to_numpy()).max())
print('Gate D — unchanged CNN + radiomics vs the E8b reference fusion (max difference):', GATE['fusion_control_vs_E8b'])

In [ ]:
# Cell 8 — E10c effects and the pre-specified rules
rng = np.random.default_rng(SEED)
def boot_ci(v):
    v = np.asarray(v, dtype=float)
    means = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(N_BOOT)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))
SCORE = {'swap': lambda p: p['p_ICC'] - p['p_HCC'], 'inject': lambda p: p['p_CHCC'], 'repair': lambda p: p['p_CHCC']}
TESTS = [   # (name, condition, patients, expected sign)
    ('slice density: HCC scans made ICC-like', 'swap', [p for p in ALL_IDS if CLS[p] == 'HCC'], +1),
    ('slice density: ICC scans made HCC-like', 'swap', [p for p in ALL_IDS if CLS[p] == 'ICC'], -1),
    ('non-contrast artefact injected (HCC, ICC)', 'inject', [p for p in ALL_IDS if CLS[p] in ('HCC', 'ICC')], +1),
    ('non-contrast artefact repaired (cHCC-CCA)', 'repair', [p for p in ALL_IDS if CLS[p] == 'CHCC'], -1),
]
P = PRED.set_index(['model', 'patient_id', 'condition'])[P3]
EFFECTS = []
for model in ['CNN', 'radiomics', 'CNN + radiomics']:
    for name, cond, ids, sign in TESTS:
        for subset, sub_ids in [('all 278', ids), ('held-out 56', [p for p in ids if p in set(test_ids)])]:
            ids_ok = [p for p in sub_ids if (model, p, cond) in P.index and (model, p, f'{cond}_smooth') in P.index and (model, p, 'control') in P.index]
            if not ids_ok:
                continue
            s_c = np.array([SCORE[cond](P.loc[(model, p, cond)]) for p in ids_ok])
            s_s = np.array([SCORE[cond](P.loc[(model, p, f'{cond}_smooth')]) for p in ids_ok])
            s_0 = np.array([SCORE[cond](P.loc[(model, p, 'control')]) for p in ids_ok])
            d = s_c - s_s
            lo, hi = boot_ci(d)
            smooth = float(np.mean(np.abs(s_s - s_0)))
            meets = bool(sign * d.mean() >= MIN_SHIFT and (lo > 0 if sign > 0 else hi < 0))
            EFFECTS.append({'model': model, 'test': name, 'subset': subset, 'n': len(ids_ok), 'expected_sign': sign,
                            'mean_shift_vs_smoothing_control': float(d.mean()), 'ci_low': lo, 'ci_high': hi,
                            'smoothing_control_mean_abs_change': smooth, 'smoothing_control_ok': smooth < MIN_SHIFT, 'meets_rule': meets})
EFFECTS = pd.DataFrame(EFFECTS)
EFFECTS.to_csv(OUT_DIR / 'E10c_effects.csv', index=False)
VERDICT = {}
for model in ['CNN', 'radiomics', 'CNN + radiomics']:
    e = EFFECTS[(EFFECTS['model'] == model) & (EFFECTS['subset'] == 'all 278')].set_index('test')
    ok = bool(e['smoothing_control_ok'].all())
    dens = e.loc[[t for t in e.index if t.startswith('slice density')], 'meets_rule']
    art = e.loc[[t for t in e.index if t.startswith('non-contrast')], 'meets_rule']
    VERDICT[model] = {
        'uses slice density': ('not interpretable (smoothing control moved the score)' if not ok else
                               'yes (both directions)' if dens.all() else 'one direction only — not claimed' if dens.any() else 'not shown'),
        'uses the non-contrast artefact': ('not interpretable (smoothing control moved the score)' if not ok else
                                           'yes (injected and repaired)' if art.all() else 'one condition only — not claimed' if art.any() else 'not shown')}
print('E10c — effects (shift of the tested score vs the smoothing control; bootstrap 95% CI):')
print(EFFECTS.round(3).to_string(index=False))
print('E10c — verdicts by the pre-specified rules:', json.dumps(VERDICT, indent=1))

In [ ]:
# Cell 9 — E10d evidence images (seeded choice: 3 cHCC-CCA and 3 HCC patients; non-contrast vs portal-venous, axial and coronal)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
EVID = OUT_DIR / 'E10d_evidence'
EVID.mkdir(exist_ok=True)
pick = np.random.default_rng(SEED)
chosen = list(pick.choice(sorted(p for p in ALL_IDS if CLS[p] == 'CHCC'), 3, replace=False)) + \
         list(pick.choice(sorted(p for p in ALL_IDS if CLS[p] == 'HCC'), 3, replace=False))
for pid in chosen:
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    for col, ph in enumerate(['P', 'C2']):
        ct = load_nii(DATA_DIR / ROWS[pid].loc[ph, 'ct_path']); tm = load_nii(DATA_DIR / ROWS[pid].loc[ph, 'mask_path']) > 0
        c = compute_center_of_mass(tm) or tuple(s // 2 for s in ct.shape)
        axes[0, col].imshow(ct[:, :, c[2]].T, cmap='gray', vmin=0, vmax=255)
        axes[0, col].contour(tm[:, :, c[2]].T, levels=[0.5], colors=['#eb6834'], linewidths=0.8)
        axes[0, col].set_title(f'{pid} ({CLS[pid]}) — {ph}, axial, slice {c[2]} of {ct.shape[2]}', fontsize=8)
        axes[1, col].imshow(ct[:, c[1], :].T, cmap='gray', vmin=0, vmax=255, aspect=1.0)
        axes[1, col].contour(tm[:, c[1], :].T, levels=[0.5], colors=['#eb6834'], linewidths=0.8)
        axes[1, col].set_title(f'{ph}, coronal in stored voxels ({ct.shape[2]} slices)', fontsize=8)
        for a in axes[:, col]:
            a.axis('off')
    fig.tight_layout(); fig.savefig(EVID / f'{pid}_{CLS[pid]}_P_vs_C2.png', dpi=200); plt.close(fig)
print('Evidence images:', sorted(p.name for p in EVID.glob('*.png')))

summary = {'run_id': RUN_ID, 'notebook': 'E10_dataset_audit_and_counterfactual_test_v2.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes,
           'gates': GATE, 'class_blocks': BLOCKS, 'e10b_verdicts': verdicts, 'e10c_verdicts': VERDICT, 'swap_factor': SWAP_FACTOR,
           'inject_ratio': INJECT_RATIO, 'min_shift': MIN_SHIFT, 'similar_threshold': SIMILAR, 'different_threshold': DIFFERENT,
           'protocol': '00 log section 24 (approved 18 Oct 2026); frozen models only; nothing trained or selected'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E10_{RUN_ID} | gates: {GATE}')
print('Class blocks (patient numbers):', BLOCKS)
print('\n--- E10a census by class ---'); print(SUMMARY_A.to_string())
print('\n--- E10b anatomy by phase and class ---'); print(SUMMARY_B.round(3).to_string())
print('E10b verdicts:', verdicts)
print('\n--- E10b non-contrast vs portal-venous, within patient ---'); print(WITHIN.round(3).to_string())
print('\n--- E10c effects ---'); print(EFFECTS.round(3).to_string(index=False))
print('\nE10c verdicts:', json.dumps(VERDICT))
print('==============================================================')

In [ ]:
# Cell 10 — Finish: make sure every result file has reached Google Drive, then shut the runtime down (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime shuts down in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()